# Metronome Compass Testing

This notebook provides utilities for testing the complex `metronome_compass` (full battle tracking) by generating seeds that produce specific outcomes.

In [1]:
%load_ext autoreload
%autoreload 2
from claytonlib.safari import advance_rng
from claytonlib.metronome_compass import precompute_path, render_path
from claytonlib.moves import _moves_by_number, resolve_move
import datetime as dt

## Expedition-style Seed Identification

Reproduce the expedition seed-ID workflow here for testing. The cell below
generates every seed within a window of a target time/delay (assuming normal
starting parameters: the Metronome user is level 7 and knows only Metronome),
computes each one's battle path, and lists their times, delays, and paths.
The following cell then interactively narrows that list to a single seed.

In [6]:
# Generate all candidate seeds within a window of a target (time, delay).
import datetime as dt
from claytonlib.times import calculate_seed
from claytonlib.metronome_compass import precompute_path, render_path
from claytonlib.moves import resolve_move

# The Metronome user's movepool BESIDES Metronome (which is always known). These
# extra moves are excluded from Metronome's roll and feed Conversion / Conversion 2
# typing, so they must match between generation (here) and the interactive cell
# below. Pass metronome_only=True to fall back to a Metronome-only user.
DEFAULT_EXTRA_MOVES = ("Fling", "Healing Wish", "Solar Beam")


def resolve_moveset(metronome_only: bool = False) -> tuple[int, ...]:
    """Move numbers for the user's non-Metronome movepool (empty if metronome_only)."""
    if metronome_only:
        return ()
    return tuple(resolve_move(name).number for name in DEFAULT_EXTRA_MOVES)


def generate_candidates_near(
    target_time: dt.datetime,
    target_delay: int,
    seconds_window: int,
    delay_window: int,
    magikarp_level: int,
    opposite_gender: bool,
    metronome_only: bool = False,
    n_turns: int = 10,
):
    """Every seed within +/-seconds_window seconds and +/-delay_window delays of
    (target_time, target_delay). The Metronome user is level 7 and, unless
    metronome_only=True, knows Metronome + DEFAULT_EXTRA_MOVES.

    Returns a list of dicts (one per unique seed) with keys:
      seed, time, delay, sec_delta, delay_delta, path, path_str
    sorted by (|delay_delta|, |sec_delta|, seed) so the target sits on top.
    """
    moveset = resolve_moveset(metronome_only)
    by_seed: dict[int, tuple] = {}
    for sec in range(-seconds_window, seconds_window + 1):
        t = target_time + dt.timedelta(seconds=sec)
        for delay in range(target_delay - delay_window, target_delay + delay_window + 1):
            if delay < 0:
                continue
            seed = calculate_seed(t, delay)
            # A seed can be reachable from several (time, delay) pairs; keep the
            # representative closest to the target.
            key = (abs(delay - target_delay), abs(sec), seed)
            if seed in by_seed and by_seed[seed][0] <= key:
                continue
            path = precompute_path(seed, magikarp_level=magikarp_level,
                                   opposite_gender=opposite_gender,
                                   moveset=moveset, n_turns=n_turns)
            by_seed[seed] = (key, {
                "seed": seed,
                "time": t,
                "delay": delay,
                "sec_delta": sec,
                "delay_delta": delay - target_delay,
                "path": path,
                "path_str": render_path(path),
            })
    candidates = [entry for _, entry in by_seed.values()]
    candidates.sort(key=lambda c: (abs(c["delay_delta"]), abs(c["sec_delta"]), c["seed"]))
    return candidates


def print_candidates(candidates, limit: int = 20):
    print(f"{len(candidates)} candidate seed(s)\n")
    print(f"  {'Seed':>10}  {'Time':>19}  {'Delay':>6}  {'dD':>4}  {'ds':>3}  Path")
    for c in candidates[:limit]:
        print(f"  0x{c['seed']:08X}  {c['time'].strftime('%Y-%m-%d %H:%M:%S')}  "
              f"{c['delay']:>6}  {c['delay_delta']:>+4}  {c['sec_delta']:>+3}  {c['path_str']}")
    if len(candidates) > limit:
        print(f"  ... and {len(candidates) - limit} more")


# --- Configure your target here ---
target_time     = dt.datetime(2025, 7, 24, 14, 55, 57)
target_delay    = 35087 # 16976 # 10026 # 7353 #6071 # 3226 #2481 # 1609 # 1881
seconds_window  = 3       # +/- X seconds
delay_window    = 4500     # +/- Y delays
magikarp_level  = 19      # < 15: Magikarp only Splashes
opposite_gender = True   # Magikarp opposite gender to the Metronome user?
metronome_only  = False   # True = Metronome-only user; False = + Fling/Healing Wish/Solar Beam

candidates = generate_candidates_near(
    target_time, target_delay, seconds_window, delay_window,
    magikarp_level=magikarp_level, opposite_gender=opposite_gender,
    metronome_only=metronome_only,
)
print_candidates(candidates)

63007 candidate seed(s)

        Seed                 Time   Delay    dD   ds  Path
  0x180E890F  2025-07-24 14:55:57   35087    +0   +0  KtkhM447h KtkhM108h KspM347 KtkhM071h KspM261- KspM404h KspM405h KspM038h KspM125- KspM040h~
  0x170E890F  2025-07-24 14:55:56   35087    +0   -1  KtkhM461_
  0x190E890F  2025-07-24 14:55:58   35087    +0   +1  KtkhM433 KspM130 Ksph KtkhM451h~ KspM397 KspM120h_
  0x160E890F  2025-07-24 14:55:55   35087    +0   -2  KtkhM008h KspM104 Ktk!M331hh KtkhM376! KspM267! Ktk-M100_
  0x1A0E890F  2025-07-24 14:55:59   35087    +0   +2  KtkhM419h KtkhM226_
  0x150E890F  2025-07-24 14:55:54   35087    +0   -3  KtkhM022h KtkhM009h KtkhM171h KtkhM370h KtkhM260h CFZM396h CFZM122h SCFZKtkhM411!~ KspM031hhh KspM090-
  0xE00E890F  2025-07-24 14:56:00   35087    +0   +3  KtkhM452h KtkhM074 KspM124h~ KspM426h KspM090- KspM241 KtkhM044h KtkhM041h KtkhM396h Ktk-M116
  0x180E890E  2025-07-24 14:55:57   35086    -1   +0  KtkhM329- Ktk!M444- KtkhM457h KspM452h KspM439h Ksp Ktk

In [10]:
# Interactively narrow `candidates` (from the cell above) to a single seed.
# Reuses the metronome_compass battle driver: an InteractiveContext walks the
# real battle turn by turn, asking what actually happened ("Magikarp used?
# (sp/tk)", "Metronome selected? (move name or M###)", "Hit, crit, or miss?",
# status prompts, ...). After each turn, candidates whose precomputed path
# diverges from what you observed are dropped.
from claytonlib.metronome_compass import (
    simulate_turn, InteractiveContext, MetronomeBattleState,
    MetronomeMove, _BATTLE_START_ADVANCES,
)
from claytonlib.moves import _moves_by_number


def narrow_candidates(candidates, magikarp_level, opposite_gender, metronome_only=False):
    moves_by_num = _moves_by_number()

    # Movepool must match the generation cell so Metronome rerolls and
    # Conversion / Conversion 2 typing line up. Metronome (118) is always implicit.
    moveset = resolve_moveset(metronome_only)
    known = frozenset(moveset)

    # One battle context drives the whole session. You answer truthfully, so its
    # state (status / locks / confusion) tracks the real battle across turns --
    # exactly the InteractiveContext the expedition metronome_compass uses.
    ctx = InteractiveContext()
    state = MetronomeBattleState()
    state.target_level = magikarp_level
    ctx.battle_state["opposite_gender"] = opposite_gender
    ctx.battle_state["state"] = state
    user_move_types = [moves_by_num[118].type_name]
    for num in moveset:
        m = moves_by_num.get(num)
        if m is not None:
            user_move_types.append(m.type_name)
    ctx.battle_state["user_move_types"] = user_move_types
    ctx.advance_unobservable(_BATTLE_START_ADVANCES)  # no-op interactively; parity w/ precompute

    def show(remaining, turn_n):
        print(f"\n{len(remaining)} / {len(candidates)} seeds remain -- next is turn {turn_n}")
        print(f"  {'Seed':>10}  {'Delay':>6}  {'dD':>4}  predicted turn {turn_n}")
        for c in remaining[:15]:
            path = c["path"]
            turn_str, move_name = "", "?"
            if len(path) >= turn_n:
                turn = path[turn_n - 1]
                turn_str = "".join(t.render() for t in turn)
                for tok in turn:
                    if isinstance(tok, MetronomeMove):
                        move_name = moves_by_num[tok.move_num].name
                        break
            print(f"  0x{c['seed']:08X}  {c['delay']:>6}  {c['delay_delta']:>+4}  "
                  f"{turn_str:<18} ({move_name})")
        if len(remaining) > 15:
            print(f"  ... and {len(remaining) - 15} more")

    remaining = list(candidates)
    turn_n = 0
    while True:
        show(remaining, turn_n + 1)
        if len(remaining) == 1:
            c = remaining[0]
            print(f"\nSeed identified: 0x{c['seed']:08X}  "
                  f"time={c['time'].strftime('%Y-%m-%d %H:%M:%S')}  delay={c['delay']}  "
                  f"dD={c['delay_delta']:+d}")
            print(f"Full path: {c['path_str']}")
            # Metronome moves remaining in the identified seed's path (turns not yet observed).
            print(f"Remaining Metronome moves (turn {turn_n + 1}+):")
            for turn_idx in range(turn_n, len(c["path"])):
                for tok in c["path"][turn_idx]:
                    if isinstance(tok, MetronomeMove):
                        print(f"  Turn {turn_idx + 1}: {moves_by_num[tok.move_num].name} (M{tok.move_num:03d})")
            return c
        if not remaining:
            print("\nNo seeds match -- check your answers or widen the window above.")
            return None

        turn_n += 1
        print(f"\n--- Turn {turn_n}: answer what happened in the battle ---")
        simulate_turn(ctx, state, moves_by_num, known, magikarp_level)
        observed = ctx.path[turn_n - 1]
        remaining = [c for c in remaining
                     if len(c["path"]) >= turn_n and c["path"][turn_n - 1] == observed]


result = narrow_candidates(candidates, magikarp_level, opposite_gender,
                           metronome_only=metronome_only)


63007 / 63007 seeds remain -- next is turn 1
        Seed   Delay    dD  predicted turn 1
  0x180E890F   35087    +0  KtkhM447h          (Grass Knot)
  0x170E890F   35087    +0  KtkhM461_          (Lunar Dance)
  0x190E890F   35087    +0  KtkhM433           (Trick Room)
  0x160E890F   35087    +0  KtkhM008h          (Ice Punch)
  0x1A0E890F   35087    +0  KtkhM419h          (Avalanche)
  0x150E890F   35087    +0  KtkhM022h          (Vine Whip)
  0xE00E890F   35087    +0  KtkhM452h          (Wood Hammer)
  0x180E890E   35086    -1  KtkhM329-          (Sheer Cold)
  0x180E8910   35088    +1  KspM073h           (Leech Seed)
  0x170E890E   35086    -1  KtkhM462h          (Crush Grip)
  0x170E8910   35088    +1  KspM245h           (Extreme Speed)
  0x190E890E   35086    -1  KtkhM315!~         (Overheat)
  0x190E8910   35088    +1  KspM368h           (Metal Burst)
  0x160E890E   35086    -1  KtkhM357           (Miracle Eye)
  0x160E8910   35088    +1  KspM106            (Harden)
  ... and 6

  Magikarp used? (sp/tk):  tk
  Tackle hit, crit, or miss? (h/!/-):  h
  Metronome selected? (move name or M###):  cross chop
  Hit, crit, or miss? (h/!/-):  -



8 / 63007 seeds remain -- next is turn 2
        Seed   Delay    dD  predicted turn 2
  0x180E8CFF   36095  +1008  KspM372h           (Assurance)
  0x150E841C   33820  -1267  KtkhM441h          (Gunk Shot)
  0x190E8E16   36374  +1287  KspM305h           (Poison Fang)
  0x190E83E1   33761  -1326  KtkhM147h          (Spore)
  0x170E824B   33355  -1732  Ktk-M226_          (Baton Pass)
  0x170E8FE7   36839  +1752  Ktk-M349           (Dragon Dance)
  0x1A0E820A   33290  -1797  Ktk-M278           (Recycle)
  0x190E96D1   38609  +3522  KspM221h~          (Sacred Fire)

--- Turn 2: answer what happened in the battle ---


  Magikarp used? (sp/tk):  sp
  Metronome selected? (move name or M###):  Assurance
  Hit, crit, or miss? (h/!/-):  h



1 / 63007 seeds remain -- next is turn 3
        Seed   Delay    dD  predicted turn 3
  0x180E8CFF   36095  +1008  KspM128hBD         (Clamp)

Seed identified: 0x180E8CFF  time=2025-07-24 14:55:57  delay=36095  dD=+1008
Full path: KtkhM238- KspM372h KspM128hBD KtkhM337!BD KspM043hBD KtkhM189h~BF KspM334 Ktk-M453h KspM459h Ksp
Remaining Metronome moves (turn 3+):
  Turn 3: Clamp (M128)
  Turn 4: Dragon Claw (M337)
  Turn 5: Leer (M043)
  Turn 6: Mud Slap (M189)
  Turn 7: Iron Defense (M334)
  Turn 8: Aqua Jet (M453)
  Turn 9: Roar Of Time (M459)


## RNG Reversing Utility

LCRNG: $state_{n+1} = (state_n \cdot 1103515245 + 24691) \pmod{2^{32}}$

Inverse: $state_n = ((state_{n+1} - 24691) \cdot 0xEEB9EB65) \pmod{2^{32}}$

In [ ]:
def reverse_rng(state: int, n: int = 1) -> int:
    """Backtrack the LCRNG n steps."""
    inv_mult = 0xEEB9EB65
    for _ in range(n):
        state = ((state - 24691) * inv_mult) & 0xFFFFFFFF
    return state


def generate_seed_for_move(
    move_name: str,
    magikarp_level: int = 2,
    n_turns: int = 3,
    verify_starting_turn: bool = False,
    find_x: int | None = None,
    criteria_substrings: list[str] | None = None,
    max_attempts: int = 1000,
):
    """Find seeds producing a specific Metronome move on Turn 1.

    Turn order: Magikarp moves first, then Metronome user.
    Advances before the Metronome roll (advance #N → seed = reverse_rng(target, N)):
      Splash (any level):  6+1+4+0+2+2 = 15  → N=16  (check rev[9]  even  → Splash)
      Tackle miss:         6+1+4+3+0+2 = 16  → N=17  (check rev[10] odd + rev[3]  >=95 → miss)
      Tackle hit:          6+1+4+3+2+2 = 18  → N=19  (check rev[12] odd + rev[5]  <95  → hit)

    Returns:
      verify_starting_turn=True  → dict mapping scenario → seed
      find_x set                 → list of up to find_x matching seeds (possibly empty)
      default                    → single matching seed (int) or None
    """
    move = resolve_move(move_name)
    if not move or not move.metronome_usable:
        raise ValueError(f"Move {move_name!r} is not metronome-usable.")

    target_num = move.number
    pool = 467
    target_val = target_num - 1

    def _check_criteria(seed: int) -> bool:
        if not criteria_substrings:
            return True
        path = precompute_path(seed, magikarp_level=magikarp_level, opposite_gender=False, n_turns=n_turns)
        rendered = render_path(path)
        return all(s in rendered for s in criteria_substrings)

    def _rev19(attempt: int):
        top16 = (target_val + pool * attempt) % 65536
        target_state = top16 << 16
        rev = [None]
        s = target_state
        for _ in range(19):
            s = ((s - 24691) * 0xEEB9EB65) & 0xFFFFFFFF
            rev.append(s)
        return rev

    if verify_starting_turn:
        results = {}
        target_count = 1 if magikarp_level < 15 else 3
        for attempt in range(max_attempts):
            rev = _rev19(attempt)
            if magikarp_level < 15:
                if 'splash' not in results and _check_criteria(rev[16]):
                    results['splash'] = rev[16]
            else:
                if 'splash' not in results and (rev[9] >> 16) % 2 == 0 and _check_criteria(rev[16]):
                    results['splash'] = rev[16]
                if 'tackle_hit' not in results and (rev[12] >> 16) % 2 == 1 and (rev[5] >> 16) % 100 < 95 and _check_criteria(rev[19]):
                    results['tackle_hit'] = rev[19]
                if 'tackle_miss' not in results and (rev[10] >> 16) % 2 == 1 and (rev[3] >> 16) % 100 >= 95 and _check_criteria(rev[17]):
                    results['tackle_miss'] = rev[17]
            if len(results) == target_count:
                break
        return results

    collected = []
    for attempt in range(max_attempts):
        rev = _rev19(attempt)
        candidates = []
        if magikarp_level < 15:
            candidates.append(rev[16])
        else:
            if (rev[9] >> 16) % 2 == 0:
                candidates.append(rev[16])
            if (rev[12] >> 16) % 2 == 1 and (rev[5] >> 16) % 100 < 95:
                candidates.append(rev[19])
            if (rev[10] >> 16) % 2 == 1 and (rev[3] >> 16) % 100 >= 95:
                candidates.append(rev[17])

        for seed in candidates:
            if _check_criteria(seed):
                if find_x is not None:
                    collected.append(seed)
                    if len(collected) >= find_x:
                        return collected
                else:
                    return seed

    return collected if find_x is not None else None


def verify_seed(seed: int, magikarp_level: int = 2, n_turns: int = 3,
                list_moves: bool = False):
    path = precompute_path(seed, magikarp_level=magikarp_level, opposite_gender=False, n_turns=n_turns)
    print(f"Seed: 0x{seed:08X}")
    print(f"Path: {render_path(path)}")
    moves = _moves_by_number()
    if list_moves:
        # Every Metronome move called along the path, in turn order.
        for turn_idx, turn in enumerate(path, 1):
            for token in turn:
                if hasattr(token, 'move_num'):
                    print(f"  Turn {turn_idx}: {moves[token.move_num].name} (M{token.move_num:03d})")
    elif path:
        for token in path[0]:
            if hasattr(token, 'move_num'):
                print(f"Move: {moves[token.move_num].name} (M{token.move_num:03d})")
                break


## Example: Seed for Flamethrower

In [ ]:
seed = generate_seed_for_move("Flamethrower")
verify_seed(seed)


## Verify a given seedd

In [ ]:
seed = generate_seed_for_move("Splash")
verify_seed(0xE70E0639, magikarp_level = 13, n_turns = 10, list_moves = True)


## Multi-Turn Path Generation

In [ ]:
seed = 0x0a7d8651
path = precompute_path(seed, magikarp_level=15, opposite_gender=False, n_turns=5)
print(f"Seed: 0x{seed:08X} (Level 15 Magikarp)")
print(f"Path: {render_path(path)}")

## All 3 Magikarp Scenarios (Level 15)

In [ ]:
seeds = generate_seed_for_move("False Swipe", magikarp_level=15, verify_starting_turn=True)
moves = _moves_by_number()
# Make sure our seed generating algorithm works no matter what the magikarp does first turn
for scenario, seed in seeds.items():
    print(f"=== {scenario} ===")
    verify_seed(seed, magikarp_level=15)
    print()


## Generating large lists of seeds

In [ ]:
seed = 0x0a7d8651
for _ in range(30):
    val = seed >> 16
    print(f"Seed: 0x{seed:08X} - Val {val}")
    seed = advance_rng(seed)

In [ ]:
## One seed per metronome-usable move (level 15 Magikarp, 6 turns)
import json as _json
import re as _re

_OUT_PATH = 'data/testpaths_2.json'

with open('claytonlib/basedata/moves.json') as _f:
    _all_moves_data = _json.load(_f)

# Preserve verification flags already recorded for these seeds (verify_paths.py
# writes them back); regenerating paths shouldn't wipe manual/auto verification.
try:
    with open(_OUT_PATH) as _f:
        _prev_verification = {_k: _v.get('verification', 0)
                              for _k, _v in _json.load(_f).items()}
except FileNotFoundError:
    _prev_verification = {}

_metronome_moves = [m for m in _all_moves_data if m['metronome_usable']]
_moves_lookup = _moves_by_number()
print(f"Generating seeds for {len(_metronome_moves)} metronome-usable moves (level 15 Magikarp, 6 turns)...\n")

_seed_map = {}
_aborted = False
for _move_data in _metronome_moves:
    _name = _move_data['name']
    _expected_num = _move_data['number']
    _seed = generate_seed_for_move(_name, magikarp_level=15, n_turns=6)
    if _seed is None:
        print(f"  WARNING: no seed found for {_name}")
        continue
    _path = precompute_path(_seed, magikarp_level=15, opposite_gender=False, n_turns=6)
    _path_str = render_path(_path)
    _move_nums = [int(m) for m in _re.findall(r'M(\d{3})', _path_str)]
    _move_names = [_moves_lookup[n].name for n in _move_nums if n in _moves_lookup]
    if not _move_nums or _move_nums[0] != _expected_num:
        _found = _moves_lookup[_move_nums[0]].name if _move_nums else 'none'
        print(f"ERROR: {_name} — expected M{_expected_num:03d} first but got {_found}. Aborting.")
        _aborted = True
        break
    _key = f"0x{_seed:08X}"
    _seed_map[_key] = {
        "verification": _prev_verification.get(_key, 0),
        "path": _path_str,
        "moves": _move_names,
    }

if not _aborted:
    with open(_OUT_PATH, 'w') as _out:
        _json.dump(_seed_map, _out, indent=2)
    print(f"Wrote {len(_seed_map)} entries to {_OUT_PATH}.")